[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Time_Series/Intro_RLS.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# RLS & Recursive Estimation

The rung between [APA](./Intro_AdFilt_APA.ipynb) and the [Kalman filter](./Intro_AdFilt_KF.ipynb): Recursive Least Squares solves the *entire* least-squares problem at every sample — exactly, recursively, without ever re-inverting a matrix — and turns out to be a Kalman filter wearing a different hat.

## 1. Pre-requisites

- [Adaptive Filtering: APA](./Intro_AdFilt_APA.ipynb) (the setup and its notation).
- [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S2 (normal equations).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

# same system-identification scenario as the APA workshop
M = 16
w_true = np.exp(-0.4*np.arange(M)) * np.cos(0.9*np.arange(M)); w_true /= np.linalg.norm(w_true)
N = 3000
from scipy import signal as sig
x = sig.lfilter([1.0], [1.0, -0.9], rng.standard_normal(N))     # correlated input (the hard case)
d = np.convolve(x, w_true)[:N] + 0.01*rng.standard_normal(N)

---
### 🕐 Session 1 of 2 — *Exponentially-Weighted Least Squares* (~35 min)
**Goal:** derive the RLS recursion from the matrix inversion lemma; implement it.
**Builds on:** [APA](./Intro_AdFilt_APA.ipynb). &nbsp; **Feeds into:** Session 2 (RLS ↔ Kalman).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: Exponentially-Weighted Least Squares</b></summary>

**Timing (~35 min).** 5 min recap of where LMS/NLMS/APA left off · 8 min the cost function and the forgetting factor · 12 min the matrix inversion lemma and the recursion · 8 min the demo · 2 min buffer.

**Board first — frame it as greed.** Draw the ladder the room has already climbed: LMS uses one sample, APA uses $K$ samples, and RLS wants *all* of them. Ask what that would cost naively — an $M \times M$ solve every sample, $O(M^3)$ — and let the room conclude it is hopeless. Then introduce the lemma as the thing that rescues it. RLS presented as "the greedy option, made affordable" lands far better than RLS presented as a formula to memorise.

**The one idea in the matrix inversion lemma.** Do not derive it in full; state what it *buys*. Each new sample adds a rank-one term $\mathbf{x}_n\mathbf{x}_n^\top$ to $R$, and the lemma says a rank-one change to a matrix produces a rank-one change to its inverse. So we never invert anything — we carry $P \approx R^{-1}$ forward and patch it. That is the entire trick, and it is worth writing "rank-one in ⟶ rank-one out" on the board as the takeaway.

**Point at the shared heartbeat.** $\mathbf{w}_n = \mathbf{w}_{n-1} + \mathbf{k}_n e_n$ is *new = old + gain × error*, the same skeleton as LMS, NLMS, APA, and (next session) Kalman. Only the gain changes. Students who see the ladder as five unrelated algorithms are carrying four times more than they need to.

**Misconception.** "λ is a learning rate." It is not — it is a *memory* parameter. $\lambda = 1$ means never forget, and weights converge to the exact least-squares solution over all history; $\lambda < 1$ discounts old data exponentially with an effective window of about $1/(1-\lambda)$ samples. At $\lambda = 0.999$ that is roughly 1000 samples. Ask the room to compute that window before you run the cell — it makes the parameter concrete instead of magical.

**Ask the room.** "Why does correlated input hurt NLMS but not RLS?" Answer: correlated input means an ill-conditioned $R$, and gradient methods crawl along its narrow valley at a rate set by the condition number $\kappa$. RLS carries $R^{-1}$, which is precisely the curvature information that rescales the valley into a bowl. This is the [Optimization S2](../Intro_Math/Optimization/Optimization.ipynb) story — gradient descent versus Newton — in an adaptive-filtering costume.

**If the demo misbehaves.** `delta` sets $P_0$ and therefore how much RLS distrusts its initial guess; too small and the first few hundred samples crawl. If the RLS curve starts flat, `delta` has been lowered. Both loops are plain Python over 3000 samples, so expect a second or two.
</details>

## 2. Solving ALL of History, Every Sample

💡 **Intuition.** LMS/NLMS/APA use a *window* of data per update. RLS is greedier: at time $n$ it wants the exact minimizer of **all** past errors, forgetting old data exponentially: $J_n = \sum_{k\le n} \lambda^{n-k} e_k^2$ with forget factor $\lambda \lesssim 1$. Naively that's a matrix solve per sample. The rescue is the **matrix inversion lemma**: a rank-one update to $R$ produces a rank-one update to $R^{-1}$ — so the inverse is *carried along* and each step costs $O(M^2)$, not $O(M^3)$.

**The recursion.** Carry $P_n \approx R_n^{-1}$:
$$\mathbf{k}_n = \frac{P_{n-1}\mathbf{x}_n}{\lambda + \mathbf{x}_n^T P_{n-1}\mathbf{x}_n} \qquad e_n = d_n - \mathbf{w}_{n-1}^T \mathbf{x}_n$$
$$\mathbf{w}_n = \mathbf{w}_{n-1} + \mathbf{k}_n e_n \qquad P_n = \lambda^{-1}\big(P_{n-1} - \mathbf{k}_n \mathbf{x}_n^T P_{n-1}\big)$$

Same heartbeat as ever — *new = old + gain × error* — but the gain $\mathbf{k}_n$ now carries the full curvature of history, so convergence is nearly immune to input correlation (no more $\kappa$ penalty from [Optimization S2](../Intro_Math/Optimization/Optimization.ipynb)).

In [2]:
def rls(x, d, M, lam=0.999, delta=100.0):
    w = np.zeros(M); P = delta*np.eye(M); e = np.zeros(len(x)); xb = np.zeros(M)
    for n_i in range(len(x)):
        xb = np.roll(xb, 1); xb[0] = x[n_i]
        Px = P @ xb
        k = Px / (lam + xb @ Px)
        e[n_i] = d[n_i] - w @ xb
        w = w + k * e[n_i]
        P = (P - np.outer(k, Px)) / lam
    return w, e

def nlms(x, d, M, mu=0.5, eps=1e-6):
    w = np.zeros(M); e = np.zeros(len(x)); xb = np.zeros(M)
    for n_i in range(len(x)):
        xb = np.roll(xb, 1); xb[0] = x[n_i]
        e[n_i] = d[n_i] - w @ xb
        w = w + mu/(eps + xb@xb) * e[n_i] * xb
    return w, e

w_rls, e_rls = rls(x, d, M)
w_nlms, e_nlms = nlms(x, d, M)

def curve(e): return 10*np.log10(np.convolve(e**2, np.ones(60)/60, "valid") + 1e-12)
plt.figure(figsize=(8, 3))
plt.plot(curve(e_nlms), label="NLMS (correlated input hurts)")
plt.plot(curve(e_rls), label="RLS (carries the inverse: barely notices)")
plt.legend(); plt.grid(True, alpha=0.3); plt.xlabel("sample"); plt.ylabel("MSE [dB]")
plt.title("Correlated input: RLS converges in ~2M samples")
plt.tight_layout(); plt.show()
print(f"weight error  RLS {np.linalg.norm(w_rls-w_true):.5f}   NLMS {np.linalg.norm(w_nlms-w_true):.5f}")

weight error  RLS 0.00137   NLMS 0.00430


/tmp/ipykernel_2040437/3591652342.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Same input, same target, same 16 taps — and RLS lands a final weight error of **0.00137** against NLMS's **0.00430**, roughly 3× closer to the truth. But the steady-state number undersells the result; the convergence curves are where the story is. RLS drops to its floor within a couple of hundred samples while NLMS is still descending thousands of samples later.

The reason is the input, and it was chosen deliberately. `x` is white noise pushed through a strongly resonant one-pole filter, so successive samples are heavily correlated and the autocorrelation matrix $R$ is badly conditioned. Gradient methods feel that directly: NLMS takes steps along the negative gradient, and in an ill-conditioned quadratic bowl the gradient points mostly *across* the narrow valley rather than along it, so progress is throttled by the condition number $\kappa$.

RLS never pays that toll, because $\mathbf{k}_n = P_{n-1}\mathbf{x}_n / (\lambda + \mathbf{x}_n^\top P_{n-1}\mathbf{x}_n)$ carries $P \approx R^{-1}$ — the curvature of the very bowl being descended. Multiplying by the inverse Hessian turns the elongated valley back into a circular one, which is Newton's method against gradient descent, exactly as in [Optimization S2](../Intro_Math/Optimization/Optimization.ipynb). "RLS converges in about $2M$ samples regardless of input colour" is the practical form of that statement, and $2M = 32$ here.

None of this is free, and the trade is worth naming now because Session 2's table formalises it: RLS costs $O(M^2)$ per sample against NLMS's $O(M)$, and it carries an $M \times M$ matrix that can drift out of symmetry over long runs. At $M = 16$ nobody cares. At $M = 1024$, in a real echo canceller, that is the difference between shipping and not.

---
### 🕐 Session 2 of 2 — *RLS ↔ Kalman* (~35 min)
**Goal:** see RLS as a Kalman filter for a static state; know the cost/robustness trade-table.
**Builds on:** Session 1; [Kalman](./Intro_AdFilt_KF.ipynb).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: RLS ↔ Kalman</b></summary>

**Timing (~35 min).** 12 min the side-by-side identification · 8 min the demo and its 3e-15 · 10 min the family portrait table · 5 min buffer. The table is the session's deliverable; leave time to actually discuss its columns rather than reading it aloud.

**Board first — the reveal.** Write the RLS recursion on the left half of the board and the Kalman equations on the right, then work across line by line asking the room to match them up. $\mathbf{k}_n$ ↔ Kalman gain, $P$ ↔ state covariance, $e_n$ ↔ innovation. Students discover the identity themselves this way, and it is far more memorable than being told. Do not put the answer up first.

**The dictionary to make explicit.** RLS *is* a Kalman filter under a specific modelling choice: the hidden state is the weight vector, $F = I$ because the weights are assumed static, $Q = 0$ because they never move, and the observation matrix $H = \mathbf{x}_n^\top$ changes every sample. That last one surprises people — the regressor plays the role of the measurement model. And $\lambda < 1$ is a back-door process noise: forgetting old data is exactly an admission that the "static" weights do drift.

**Misconception.** "So RLS and Kalman are just the same thing, and the distinction is pedantic." No — the identity holds *only* for a static state. Kalman is the strictly more general object: it accepts any $F$ and $Q$, so it can track a state that moves according to a known model, and it tells you your uncertainty as a calibrated covariance rather than as a bookkeeping matrix. RLS is the corner of that space where you have declined to write a state model. Ask what you would use if the echo path were known to drift like a random walk — the answer is Kalman with $Q \neq 0$, and $\lambda$ is only a crude scalar stand-in for it.

**Ask the room.** "The demo agrees to 3e-15. Is that a good test?" It is a genuinely strong one — that is floating-point round-off over 3000 sequential updates, so the two recursions are algebraically identical rather than merely similar. Contrast with a test that reported agreement to 1e-3, which would only show two algorithms landing in the same neighbourhood. Getting students to distinguish "same answer" from "same algorithm" is worth the detour.

**Working the table.** Do not read it row by row. Pose scenarios and let the room pick a column: a 1024-tap echo canceller on a phone (NLMS — $O(M^2)$ is unaffordable), a 16-tap channel equaliser with heavily correlated input (RLS), a satellite whose orbit follows known dynamics (Kalman, and the state model is the whole point). The trade-offs stick when they are attached to a decision.

**Flag the numerical caveat.** The starred footnote matters more than its size suggests: the textbook $P$ update loses symmetry and positive-definiteness over long runs, and production code uses QR or square-root forms. Same lesson as second-order sections in [Filter Design](../Intro_DSP/Filter_Design.ipynb) — factor, don't invert. If anyone plans to deploy this, that is the sentence they need.
</details>

## 3. The Identification

💡 **Intuition.** Stare at the RLS recursion next to the [Kalman equations](./Intro_AdFilt_KF.ipynb): they are the *same algorithm*. Model the weights as a **static hidden state** ($F = I$, $Q = 0$) observed through $d_n = \mathbf{x}_n^T \mathbf{w} + v_n$ (so $H = \mathbf{x}_n^T$ changes every step): the Kalman gain becomes $\mathbf{k}_n$, the covariance $P$ is RLS's $P$, and $\lambda < 1$ plays the role of process noise — a confession that the 'static' weights actually drift. One framework, three names: RLS (filtering), recursive least squares (statistics), Kalman with random regressors (control).

In [3]:
# Verify the identification numerically: Kalman-with-static-state ≡ RLS (λ=1)
def kalman_static(x, d, M, R_meas=1.0, P0=100.0):
    w = np.zeros(M); P = P0*np.eye(M); xb = np.zeros(M)
    for n_i in range(len(x)):
        xb = np.roll(xb, 1); xb[0] = x[n_i]
        S = xb @ P @ xb + R_meas
        k = P @ xb / S
        w = w + k * (d[n_i] - w @ xb)
        P = P - np.outer(k, xb @ P)
    return w

w_kal = kalman_static(x, d, M)
w_rls1, _ = rls(x, d, M, lam=1.0, delta=100.0)
print("max |w_Kalman − w_RLS(λ=1)| =", np.abs(w_kal - w_rls1).max())

max |w_Kalman − w_RLS(λ=1)| = 3.1580640880157773e-15


**What just happened.** Two functions written from different starting points — one derived from the matrix inversion lemma, one from Bayesian state estimation — produce weight vectors agreeing to **3.2e-15**.

That number deserves to be read carefully, because it is a much stronger claim than "these methods perform similarly." It is machine epsilon accumulated over 3000 sequential updates, each involving a matrix–vector product and a division. Two algorithms that merely converged to the same optimum would agree to maybe 1e-6 and would disagree *along the way*; these agree at every step, to round-off. RLS with $\lambda = 1$ and a Kalman filter over a static state are not analogous, not asymptotically equivalent — they are the same recursion with different variable names.

Line them up and the dictionary is exact: the Kalman gain $P\mathbf{x}/(\mathbf{x}^\top P \mathbf{x} + R)$ is RLS's $\mathbf{k}_n$ with the measurement noise $R$ playing $\lambda$'s part; the covariance update is RLS's $P$ update; the innovation $d_n - \mathbf{w}^\top\mathbf{x}_n$ is the a-priori error. The modelling assumptions that produce this are $F = I$ and $Q = 0$ — the hidden state is the weight vector, and it is assumed never to move.

**What the identity does not say.** Kalman remains strictly the more general object, and the equivalence is confined to the static-state corner of it. Give the state a dynamics model $F$ and a process noise $Q$ and you can track weights that genuinely move, with an uncertainty that means something in probability rather than serving as bookkeeping. Seen from there, RLS's forgetting factor $\lambda$ is a one-scalar approximation to a full $Q$ — cheap, effective, and blunt, since it discounts every direction of the weight space at the same rate whether or not that matches how the system actually drifts.

The ladder from LMS to Kalman is therefore one algorithm at five levels of self-knowledge, and every rung shares the heartbeat *new = old + gain × error*. What changes is only how much the gain knows: nothing (LMS), the input power (NLMS), a small window (APA), all of history's curvature (RLS), or an explicit model of how the world moves (Kalman).

### The Family Portrait

| | LMS | NLMS | APA-$K$ | RLS | Kalman |
|---|---|---|---|---|---|
| Cost/sample | $O(M)$ | $O(M)$ | $O(K^2M)$ | $O(M^2)$ | $O(M^2)$+model |
| Colored-input speed | ✗ | ✗ | ○ | ✓ | ✓ |
| Tracks drifting systems | ✓ | ✓ | ✓ | via $\lambda$ | via $Q$ (principled) |
| Needs a state model | – | – | – | – | **yes** |
| Numerical fragility | robust | robust | mild | $P$ can lose symmetry* | same, use square-root forms |

\*Production RLS/Kalman uses QR/square-root updates — the [SOS lesson](../Intro_DSP/Filter_Design.ipynb) again: factor, don't invert.

## 4. Conclusion

RLS = exact least squares carried recursively via the matrix inversion lemma = Kalman filtering a static state. The whole adaptive-filtering ladder is one algorithm at increasing levels of self-knowledge.

---
## Where next

- [Beyond Kalman](./Beyond_Kalman.ipynb) — when the state *moves nonlinearly*.
- [Kernel Methods](../Intro_Mach_Learn/Kernel_Methods.ipynb) — KRLS: this recursion in a feature space.